In [ ]:
import pandas as pd
import numpy as np
import sklearn as skl
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.cluster import KMeans
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from challenge_spotter import config as cfg
from challenge_spotter.auxiliars import get_lgbm_baseline,get_random_forest_baseline,get_xgb_baseline,get_pipeline,get_features_dict,run_temporal_cv,prepare_data,get_linear_regresion_baseline,get_pipeline_target_log_transform,show_results
import optuna
from sklearn.linear_model import Ridge


train_df= pd.read_parquet(cfg.DATA_DIR / "train.parquet")
test_df= pd.read_parquet(cfg.DATA_DIR / "test.parquet")

train_df= train_df.sort_values(by="date",ascending=True).reset_index(drop=True)
features_dict= get_features_dict()


c:\Users\kuroc\OneDrive\Escritorio\challenge_spotter\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#The best baseline
X_train , Y_train = prepare_data(train_df)
linear_regresor= get_linear_regresion_baseline()
pipeline= get_pipeline(linear_regresor,features_dict,scale_features=False)
run_temporal_cv(pipeline,X_train,Y_train)

results per fold:
  Fold 1: 524.1553
  Fold 2: 514.5565
  Fold 3: 559.8852
  Fold 4: 677.2323
  Fold 5: 649.8200
------
  RMSE Mean: 585.1299
  Std: 66.3377
  Min / Max: [514.5565, 677.2323]


array([524.15531515, 514.55646753, 559.88522623, 677.232269  ,
       649.82000048])

In [4]:
#the performance drop applying the log transform to the target ussing the linear model.
X_train , Y_train = prepare_data(train_df)
linear_regresor_log= get_linear_regresion_baseline()
pipeline_linear_regresor_log= get_pipeline(linear_regresor_log,features_dict,scale_features=False)
pipeline_wraper_log_transform= get_pipeline_target_log_transform(pipeline_linear_regresor_log)
run_temporal_cv(pipeline_wraper_log_transform,X_train,Y_train)


results per fold:
  Fold 1: 590.2006
  Fold 2: 1463.2355
  Fold 3: 672.8090
  Fold 4: 743.9944
  Fold 5: 715.1003
------
  RMSE Mean: 837.0679
  Std: 317.3481
  Min / Max: [590.2006, 1463.2355]


array([ 590.20056974, 1463.23548624,  672.8089681 ,  743.99442666,
        715.10029609])

In [ ]:
#the random forest had the worst performance in the baseline testing, but improve a lot ussing log transform and some features.
#still being worst than the linear model
X_train , Y_train = prepare_data(train_df)
random_forest= get_random_forest_baseline()
rf_pipeline= get_pipeline(random_forest,features_dict)
rf_pipeline_log_transform= get_pipeline_target_log_transform(rf_pipeline)
run_temporal_cv(rf_pipeline_log_transform,X_train,Y_train)

results per fold:
  Fold 1: 600.4742
  Fold 2: 645.4661
  Fold 3: 631.4189
  Fold 4: 753.5473
  Fold 5: 701.8586
------
  RMSE Mean: 666.5530
  Std: 54.5160
  Min / Max: [600.4742, 753.5473]


array([-600.47423812, -645.46610866, -631.41893272, -753.54732943,
       -701.85862617])

In [ ]:
#consistenly better than RF, worst than the linear regressor. Also improve a little bit when log transform is applied to the target.
X_train , Y_train = prepare_data(train_df)
xgb_model= get_xgb_baseline()
pipeline_xgb= get_pipeline(xgb_model,features_dict)
pipeline_xgb_log_transform= get_pipeline_target_log_transform(pipeline_xgb)
run_temporal_cv(pipeline_xgb,X_train,Y_train)

results per fold:
  Fold 1: 546.7361
  Fold 2: 596.3372
  Fold 3: 563.2180
  Fold 4: 685.4903
  Fold 5: 656.3537
------
  RMSE Mean: 609.6271
  Std: 53.3372
  Min / Max: [546.7361, 685.4903]


array([-546.7361271 , -596.33716838, -563.21800857, -685.49028905,
       -656.35371732])

In [ ]:
#the second best one. Matching the perfromance of the linear model.
X_train , Y_train = prepare_data(train_df)
lgbm_model= get_lgbm_baseline()
lgbm_pipeline= get_pipeline(lgbm_model,features_dict)
lgbm_log_transform= get_pipeline_target_log_transform(lgbm_pipeline)
run_temporal_cv(lgbm_log_transform,X_train,Y_train)

results per fold:
  Fold 1: 523.5042
  Fold 2: 528.9194
  Fold 3: 559.8921
  Fold 4: 681.8092
  Fold 5: 654.5321
------
  RMSE Mean: 589.7314
  Std: 65.8057
  Min / Max: [523.5042, 681.8092]


array([-523.50415567, -528.9194492 , -559.89212805, -681.80920816,
       -654.53210551])

With that explored, i proceed to search optimal hyperparams for the 2 best models. (Ridge and LGBM).

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_objective_ridge(X_train, Y_train, features_dict):
    def objective_ridge(trial):

        alpha = trial.suggest_float("alpha", 1e-3, 1e4, log=True)

        clusters = trial.suggest_int("clusters", 1, 20, log=True)

        
        linear_model = Ridge(alpha=alpha, random_state=42)

        pipeline = get_pipeline(linear_model,features_dict,scale_features=True,n_clusters=clusters)

        score= run_temporal_cv(pipeline,X_train,Y_train,show_logs=False,eval_metric="MAE")
        
        return score.mean()
    return objective_ridge

X_train , Y_train= prepare_data(train_df)
objective_fn = create_objective_ridge(X_train, Y_train, features_dict)
study_ridge = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=42))
study_ridge.optimize(objective_fn, n_trials=200)

print(f"best score: {study_ridge.best_value:.4f}")
for param, value in study_ridge.best_params.items():
    print(f"  - {param}: {value}")

best score: 630.0677
  - alpha: 0.17786083799347527
  - clusters: 6


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_objective_ridge(X_train, Y_train, features_dict):
    def objective_ridge(trial):

        n_clusters = trial.suggest_int("n_clusters", 2, 20)
        learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
        num_leaves = trial.suggest_int("num_leaves", 15, 255)
        min_child_samples = trial.suggest_int("min_child_samples", 10, 100)
        n_estimators = trial.suggest_int("n_estimators", 100, 1000)
        subsample = trial.suggest_float("subsample", 0.6, 1.0)
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)

        lgbm_model = LGBMRegressor(
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        max_depth=-1,
        min_child_samples=min_child_samples,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        )

        lgbm_op_pipeline = get_pipeline(lgbm_model,features_dict,scale_features=True,n_clusters=n_clusters)
        lgbm_op_pipeline_log= get_pipeline_target_log_transform(lgbm_op_pipeline)
        score= run_temporal_cv(lgbm_op_pipeline_log,X_train,Y_train,show_logs=False,eval_metric="MAE")
        return score.mean()
    return objective_ridge

X_train , Y_train= prepare_data(train_df)
objective_fn = create_objective_ridge(X_train, Y_train, features_dict)
study_ridge = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=42))
study_ridge.optimize(objective_fn, n_trials=200)

print(f"best results :{study_ridge.best_value:.4f}")
for param, value in study_ridge.best_params.items():
    print(f"  - {param}: {value}")

best RMSE :117.1185
  - n_clusters: 19
  - learning_rate: 0.013509291448953806
  - num_leaves: 15
  - min_child_samples: 72
  - n_estimators: 847
  - subsample: 0.8689120872516707
  - colsample_bytree: 0.7694118918514066


best RMSE :583.5129
  - n_clusters: 3
  - learning_rate: 0.04085923134532415
  - num_leaves: 64
  - max_depth: 2
  - min_child_samples: 56
  - n_estimators: 711
  - subsample: 0.9580712658563583
  - colsample_bytree: 0.8089081041365123

Finally i will run CV with the optimal hyperparams founded and check against test. 

In [3]:
linear_optuna_model = Ridge(alpha=17.269978740046007, random_state=42)
X_train , Y_train = prepare_data(train_df)
ridge_optuna_pipeline= get_pipeline(linear_optuna_model,features_dict,n_clusters=5,scale_features=True)
run_temporal_cv(ridge_optuna_pipeline,X_train,Y_train,eval_metric="MAE")

X_test, Y_test= prepare_data(test_df)

ridge_optuna_pipeline.fit(X_train,Y_train)
ridge_predictions_oos= ridge_optuna_pipeline.predict(X_test)
show_results(y_test=Y_test, y_predicted= ridge_predictions_oos)

results per fold:
  Fold 1: 134.3526
  Fold 2: 123.6995
  Fold 3: 126.2971
  Fold 4: 137.5211
  Fold 5: 132.0989
------
  MAE Mean: 130.7938
  Std: 5.1024
  Min / Max: [123.6995, 137.5211]
----------------
  Test MAE:  145.6557
  Test RMSE: 647.9631


In [5]:
pd.Series(ridge_predictions_oos).max()

6500.832926343854

In [ ]:
audit_df = X_test.copy()
audit_df["real_cost"] = Y_test
audit_df["predicted_cost"] = ridge_predictions_oos
audit_df["error"] = np.abs(audit_df["real_cost"] - audit_df["predicted_cost"])

worst_predictions = audit_df.sort_values(by="error", ascending=False)
cols_to_inspect = [
    "pickup",
    "delivery",
    "equipment",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "real_cost",
    "predicted_cost",
    "error",
]

print(worst_predictions[cols_to_inspect].head(15).to_string())
print(f"\nMAE OOS: {audit_df['error'].mean():.2f}")
print(f"Max Error OOS: {audit_df['error'].max():.2f}")

top_20 = worst_predictions[cols_to_inspect].head(30)
p99 = audit_df["real_cost"].quantile(0.99)
top_error_no_outliers= (top_20["real_cost"] < 7000).sum()
print(f"ammong the top 30 worst predictions, the ammount of cases that are not outliers is {top_error_no_outliers}")


            pickup       delivery equipment  distance   weight  market_index  quote_signal  real_cost  predicted_cost         error
1462     Baltimore  Oklahoma City    Reefer    1760.3  29521.0       1.00464       2.28753   20132.37     3745.860885  16386.509115
1537    Montgomery         Fresno   Dry Van    2283.4  32447.0       0.98502       1.93645   20361.89     4397.315653  15964.574347
4905     Milwaukee    Bakersfield   Dry Van    1893.0  30760.0       1.08079       2.22972   17514.11     3566.896035  13947.213965
7579      Columbia         Tucson   Flatbed    2023.4  34710.0       0.89822       2.06797   17893.17     4008.188795  13884.981205
8664        Boston    Bakersfield    Reefer    2979.1  30585.0       0.86501       2.09472   19110.30     5553.506112  13556.793888
1688    Montgomery        El Paso   Dry Van    1639.9  38363.0       0.96139       1.92501   14949.25     3294.473404  11654.776596
7882        Toledo    Albuquerque    Reefer    1684.3  31536.0       0.98632

In [10]:
lgbm_model_tuned = LGBMRegressor(
    learning_rate=0.013509291448953806,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=5,
    n_estimators=847,
    subsample=0.8689120872516707,
    colsample_bytree=0.7694118918514066,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,)

X_train , Y_train = prepare_data(train_df)
lgbm_optuna_pipeline= get_pipeline(lgbm_model_tuned,features_dict,n_clusters=5)
log_pipeline= get_pipeline_target_log_transform(lgbm_optuna_pipeline)

run_temporal_cv(lgbm_optuna_pipeline,X_train,Y_train,eval_metric="MAE")



X_test, Y_test= prepare_data(test_df)

lgbm_optuna_pipeline.fit(X=X_train,y=Y_train)
predicts_lgbm_oos=lgbm_optuna_pipeline.predict(X_test)
show_results(y_test=Y_test, y_predicted= predicts_lgbm_oos)


results per fold:
  Fold 1: 207.3079
  Fold 2: 233.7374
  Fold 3: 124.0716
  Fold 4: 137.0565
  Fold 5: 122.6022
------
  MAE Mean: 164.9551
  Std: 46.4076
  Min / Max: [122.6022, 233.7374]
----------------
  Test MAE:  145.4919
  Test RMSE: 649.9105


conclusions: The linear model consistenly matched the performance of more complex, tuned models, making it the best option in terms of complexity / performance.
Additionally  the metrics (MAE and RMSE) have a huge gap this is completly caused because RMSE are sensitive to outliers. As shown previously The top 30 worst predictions are all above the percentile 99% in the target scale. And having 130 values over 7000, seems extremely hard to improve the performance in that segment without degrading  the performance in the other 99% of the dataset.